# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****3f08
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

C:\ceshiren\lca-langchainV1-essentials\python_local\.venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
from langchain.chat_models import init_chat_model

# initialize a commercial chat model (e.g., "openai:gpt-5") or use a local
# model (e.g., "ollama:gpt-oss")
model = init_chat_model("ollama:gpt-oss:latest", temperature=0)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [8]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (db05fc1b-0e34-49f2-ad6a-526ac93b4dd2)
 Call ID: db05fc1b-0e34-49f2-ad6a-526ac93b4dd2
  Args:
    query: PRAGMA table_info(invoices);
================================= Tool Message =================================
Name: execute_sql


================================== Ai Message ==================================
Tool Calls:
  execute_sql (b23f559f-0701-4ae7-935e-da64591799bd)
 Call ID: b23f559f-0701-4ae7-935e-da64591799bd
  Args:
    query: SELECT * FROM invoices LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT * FROM invoices LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
=============

In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (0401a03e-4be8-4e59-9f94-2360805abb49)
 Call ID: 0401a03e-4be8-4e59-9f94-2360805abb49
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

Could you let me know which titles you’re referring to? For example, are you interested in track titles, album titles, playlist titles, or something else?


## Add memory

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [12]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (94fcce2f-e38c-40e1-bde2-2ab34210e060)
 Call ID: 94fcce2f-e38c-40e1-bde2-2ab34210e060
  Args:
    query: PRAGMA table_info(invoices);
================================= Tool Message =================================
Name: execute_sql


================================== Ai Message ==================================
Tool Calls:
  execute_sql (93b353c2-093c-4669-a28e-1a65cf3fdb4a)
 Call ID: 93b353c2-093c-4669-a28e-1a65cf3fdb4a
  Args:
    query: SELECT * FROM invoices LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT * FROM invoices LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
=============

In [13]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (15835de7-2464-46b8-9bfd-b6e0738f1c92)
 Call ID: 15835de7-2464-46b8-9bfd-b6e0738f1c92
  Args:
    query: SELECT t.Title FROM Track t JOIN InvoiceLine il ON t.TrackId = il.TrackId WHERE il.InvoiceId = 374;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: t.Title
[SQL: SELECT t.Title FROM Track t JOIN InvoiceLine il ON t.TrackId = il.TrackId WHERE il.InvoiceId = 374;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (ebd43dba-89c9-4dd9-9bed-89eec0acf6e2)
 Call ID: ebd43dba-89c9-4dd9-9bed-89eec0acf6e2
  Args:
    query: PRAGMA table_info(Track);
====================

## Try your own queries
Now that there is memory, check the agents recall!

In [14]:
question = "what is the longest title?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

what is the longest title?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (04ca3810-d135-413a-916f-0287e3b6cbf7)
 Call ID: 04ca3810-d135-413a-916f-0287e3b6cbf7
  Args:
    query: SELECT Name, LENGTH(Name) AS len FROM Track WHERE TrackId IN (SELECT TrackId FROM InvoiceLine WHERE InvoiceId=374) ORDER BY len DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[('My Friend Of Misery', 19)]
================================== Ai Message ==================================

The longest title on that invoice is **“My Friend Of Misery”** (19 characters).
